In [ ]:
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from zipfile import ZipFile


In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def download_if_needed(url, path):
    if not path.exists():
        print(f"Downloading {path.name}...")
        r = requests.get(url)
        r.raise_for_status()
        path.write_bytes(r.content)
    else:
        print(f"{path.name} already exists.")

def extract_if_needed(zip_path, extract_folder):
    if not extract_folder.exists():
        print(f"Extracting {zip_path.name}...")
        with ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_folder)
    else:
        print(f"{extract_folder.name} already exists.")


loc_url = 'https://data.london.gov.uk/download/24rz6/77d9b319-931e-4090-bf8e-f578938bd352/LSOA2011%20AvPTAI2015.csv'
loc_path =  DATA_DIR / "location.csv"
crime_url = 'https://data.london.gov.uk/download/exy3m/vm7/MPS%20LSOA%20Level%20Crime%20(Historical).csv'
crime_path = DATA_DIR / "crime.csv"
deprivation_url = "https://assets.publishing.service.gov.uk/media/5dc407b440f0b6379a7acc8d/File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv"
deprivation_path = DATA_DIR / "deprivation.csv" 
house_url = "https://www.ons.gov.uk/file?uri=/peoplepopulationandcommunity/housing/datasets/medianpricepaidbylowerlayersuperoutputareahpssadataset46/current/hpssadataset46medianpricepaidforresidentialpropertiesbylsoa.zip"
house_zip = DATA_DIR / "house_prices.zip"
house_path = DATA_DIR / "house_prices"
download_if_needed(loc_url,loc_path)
download_if_needed(crime_url,crime_path)
download_if_needed(deprivation_url,deprivation_path)
download_if_needed(house_url, house_zip)
extract_if_needed(house_zip, house_path)

In [79]:
excel_file = next(house_path.glob("*.xls"))
loc_df = pd.read_csv('data/location.csv')
crime_df = pd.read_csv('data/crime.csv')
IoD_df = pd.read_csv('data/deprivation.csv')
house_df = pd.read_excel(
    excel_file,
    sheet_name="1a",
    header=5
)

In [80]:
print(loc_df.columns)
print(crime_df.columns)
print(IoD_df.columns)
print(house_df.columns)

Index(['LSOA2011', 'AvPTAI2015', 'PTAL', 'PTAIHigh', 'PTAILow'], dtype='str')
Index(['LSOA Code', 'LSOA Name', 'Borough', 'Group', 'SubGroup', '202007',
       '202008', '202009', '202010', '202011', '202012', '202101', '202102',
       '202103', '202104', '202105', '202106', '202107', '202108', '202109',
       '202110', '202111', '202112', '202201', '202202', '202203', '202204',
       '202205', '202206', '202207', '202208', '202209', '202210', '202211',
       '202212', '202301', '202302', '202303', '202304', '202305', '202306',
       '202307', '202308', '202309', '202310', '202311', '202312', '202401',
       '202402', '202403', '202404', '202405', '202406'],
      dtype='str')
Index(['LSOA code (2011)', 'LSOA name (2011)',
       'Local Authority District code (2019)',
       'Local Authority District name (2019)',
       'Index of Multiple Deprivation (IMD) Score',
       'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)',
       'Index of Multiple Deprivation

In [82]:
 
loc_df = loc_df[['LSOA2011', 'AvPTAI2015', 'PTAL']]
crime_df = crime_df[['LSOA Code', 'Group','202201', '202202', '202203', '202204', '202205', '202206',
       '202207', '202208', '202209', '202210', '202211', '202212']]
IoD_df = IoD_df[['LSOA code (2011)', 'LSOA name (2011)',
       'Local Authority District code (2019)',
       'Local Authority District name (2019)',
       'Index of Multiple Deprivation (IMD) Score']]
house_df = house_df[['Local authority code', 'Local authority name', 'LSOA code',
       'LSOA name', 'Year ending Dec 2021', 'Year ending Mar 2022',
       'Year ending Jun 2022', 'Year ending Sep 2022', 'Year ending Dec 2022']]


In [ ]:
# print(loc_df.head())
# print(crime_df.head())
# print(IoD_df.head())
# print(house_df.head())

In [ ]:
house_london = house_df[house_df["Local authority code"].astype(str).str.startswith("E09", na=False)].copy()
price_cols = [
    "Year ending Dec 2022",
    "Year ending Sep 2022",
    "Year ending Jun 2022",
    "Year ending Mar 2022",
    "Year ending Dec 2021"
]
house_london[price_cols] = house_london[price_cols].replace(":", pd.NA)

house_london[price_cols] = house_london[price_cols].apply(pd.to_numeric)

# Create the final house price column by taking the first available value
house_london["house_price_2022"] = house_london[price_cols].bfill(axis=1).iloc[:, 0]

# Convert each row to True/False for missing values. idxmax(axis=1) returns the name of the first column containing True.
house_london["price_source"] = house_london[price_cols].notna().idxmax(axis=1)

# idxmax returns the first column even when all values are missing, so we need to label those rows correctly
house_london.loc[house_london["house_price_2022"].isna(),"price_source"] = "Missing"

print(house_london["price_source"].value_counts())
house_london=house_london.drop(columns = price_cols)


price_source
Year ending Dec 2022    4496
Missing                  111
Year ending Mar 2022      86
Year ending Sep 2022      64
Year ending Jun 2022      43
Year ending Dec 2021      35
Name: count, dtype: int64
111


In [84]:
crime_months = [
    "202201", "202202", "202203", "202204",
    "202205", "202206", "202207", "202208",
    "202209", "202210", "202211", "202212"
]

crime_df[crime_months] = crime_df[crime_months].apply(pd.to_numeric,errors="coerce").fillna(0)

crime_df["crime_total_2022"] = crime_df[crime_months].sum(axis=1)
total_crime_df = crime_df.drop(columns = crime_months).copy()


In [85]:
crime_by_lsoa = (
    total_crime_df
    .pivot_table(
        index="LSOA Code",
        columns="Group",
        values="crime_total_2022",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)


# Remove the name added above the pivoted columns
crime_by_lsoa.columns.name = None

# Add total crime across all crime-category columns
crime_by_lsoa["total_crime_2022"] = (crime_by_lsoa.drop(columns="LSOA Code").sum(axis=1))


In [ ]:
look_up_url = 'https://open-geography-portalx-ons.hub.arcgis.com/api/download/v1/items/b684a0dbf786473f9563ec0616da2f8b/csv?layers=0'
look_up_path = DATA_DIR / "lookup.csv" 
download_if_needed(look_up_url,look_up_path)
lookup_df = pd.read_csv('data/lookup.csv')
lookup_df = lookup_df[['LSOA11CD','LSOA21CD']]


In [86]:
crime_2011 = crime_by_lsoa.merge(
    lookup_df,
    left_on="LSOA Code",
    right_on="LSOA21CD",
    how="inner"
)

In [87]:
house_crime = house_london.merge(
    crime_2011,
    left_on="LSOA code",
    right_on="LSOA11CD",
    how="left",
    validate="one_to_one"
)

house_crime = house_crime.drop(columns=["LSOA11CD","LSOA21CD","LSOA Code"])

In [88]:
print(f"House price LSOAs: {len(house_london)}")
print(f"Crime LSOAs after lookup: {len(crime_2011)}")
print(f"Merged dataset: {len(house_crime)}")
print(f"Unmatched house-price LSOAs: {house_crime['total_crime_2022'].isna().sum()}")


missing = house_crime[house_crime["total_crime_2022"].isna()]

missing["Local authority name"].value_counts()

House price LSOAs: 4835
Crime LSOAs after lookup: 4832
Merged dataset: 4835
Unmatched house-price LSOAs: 3


Local authority name
City of London    3
Name: count, dtype: int64

In [90]:
deprivation_transport = IoD_df.merge(
    loc_df,
    left_on="LSOA code (2011)",
    right_on="LSOA2011",
    how="left",
    validate="one_to_one"
)

# Drop the duplicate transport LSOA code
deprivation_transport = deprivation_transport.drop(columns=['Local Authority District code (2019)','Local Authority District name (2019)','LSOA name (2011)','LSOA2011'])
deprivation_transport = deprivation_transport.rename(
    columns={
        'Index of Multiple Deprivation (IMD) Score': 'IMD score',
        "LSOA code (2011)": "LSOA code"
    }
)



In [93]:
final_df = house_crime.merge(
    deprivation_transport,
    on="LSOA code",
    how="left",
    validate="one_to_one"
)

In [94]:
output_path = DATA_DIR / "final_dataset.csv"
final_df.to_csv(output_path, index=False)

print(final_df.shape)
print(final_df.columns)

final_df.head()

(4835, 20)
Index(['Local authority code', 'Local authority name', 'LSOA code',
       'LSOA name', 'house_price_2022', 'price_source',
       'ARSON AND CRIMINAL DAMAGE', 'BURGLARY', 'DRUG OFFENCES',
       'MISCELLANEOUS CRIMES AGAINST SOCIETY', 'POSSESSION OF WEAPONS',
       'PUBLIC ORDER OFFENCES', 'ROBBERY', 'THEFT', 'VEHICLE OFFENCES',
       'VIOLENCE AGAINST THE PERSON', 'total_crime_2022', 'IMD score',
       'AvPTAI2015', 'PTAL'],
      dtype='str')


,Local authority code,Local authority name,LSOA code,LSOA name,house_price_2022,price_source,ARSON AND CRIMINAL DAMAGE,BURGLARY,DRUG OFFENCES,MISCELLANEOUS CRIMES AGAINST SOCIETY,POSSESSION OF WEAPONS,PUBLIC ORDER OFFENCES,ROBBERY,THEFT,VEHICLE OFFENCES,VIOLENCE AGAINST THE PERSON,total_crime_2022,IMD score,AvPTAI2015,PTAL
0,E09000001,City of London,E01000001,City of London 001A,880000.0,Year ending Dec 2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.208,69.8233,6b
1,E09000001,City of London,E01000002,City of London 001B,850000.0,Year ending Dec 2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.143,83.7820,6b
2,E09000001,City of London,E01000003,City of London 001C,540000.0,Year ending Dec 2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.402,41.7417,6b
3,E09000001,City of London,E01000005,City of London 001E,NaN,Missing,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,28.652,85.8893,6b
4,E09000001,City of London,E01032739,City of London 001F,756250.0,Year ending Dec 2022,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,13.584,111.4790,6b


In [95]:
print("Missing house prices:", final_df["house_price_2022"].isna().sum())

print("Missing crime:", final_df["total_crime_2022"].isna().sum())

print("Missing IMD:", final_df["IMD score"].isna().sum())

print("Missing PTAI:", final_df["AvPTAI2015"].isna().sum())

Missing house prices: 111
Missing crime: 3
Missing IMD: 0
Missing PTAI: 0
